# 💻 Unidad 4: Material Complementario - Práctica
## Módulo 03 - A/B Testing y Experimentación
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de la Práctica

En esta práctica vas a:

1. ✅ Simular un experimento A/B
2. ✅ Calcular tamaño de muestra necesario
3. ✅ Realizar test de significancia estadística
4. ✅ Calcular poder estadístico
5. ✅ Interpretar resultados y tomar decisiones

---

### 📋 Ejercicios

1. **Ejercicio 1**: Cálculo de tamaño de muestra
2. **Ejercicio 2**: Simular experimento A/B
3. **Ejercicio 3**: Test de significancia
4. **Ejercicio 4**: Análisis de poder estadístico

---

### ⏱️ Duración Estimada: 60 minutos

## 🛠️ Setup: Instalación de Librerías

Instalamos las librerías necesarias para A/B testing:

In [0]:
# Instalar librerías para A/B testing
%pip install scipy statsmodels

print("✅ Librerías instaladas")

In [0]:
# Imports
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import zt_ind_solve_power
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 📊 Ejercicio 1: Cálculo de Tamaño de Muestra

**Objetivo**: Determinar cuántos usuarios necesitamos para detectar un efecto mínimo

**Parámetros**:
* Tasa de conversión base: 10%
* Mínimo efecto detectable (MDE): 1% (lift relativo del 10%)
* Nivel de significancia (α): 0.05
* Poder estadístico (1-β): 0.80

**Fórmula**: Usamos la fórmula de tamaño de muestra para test de proporciones

In [0]:
print("📊 Ejercicio 1: Cálculo de tamaño de muestra\n" + "="*60)

# Parámetros
p1 = 0.10  # Tasa de conversión baseline (Grupo A)
p2 = 0.11  # Tasa de conversión objetivo (Grupo B) - 10% lift relativo
alpha = 0.05  # Nivel de significancia (error tipo I)
power = 0.80  # Poder estadístico (1 - error tipo II)

print("Parámetros del experimento:")
print(f"  Conversión baseline (A): {p1:.1%}")
print(f"  Conversión objetivo (B): {p2:.1%}")
print(f"  Lift absoluto: {(p2-p1):.1%}")
print(f"  Lift relativo: {((p2-p1)/p1):.1%}")
print(f"  α (significancia): {alpha}")
print(f"  Poder (1-β): {power}")

# Calcular effect size (Cohen's h)
effect_size = 2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1)))

# Calcular tamaño de muestra por grupo
n_per_group = zt_ind_solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative='two-sided'
)

print(f"\n📊 Resultados:")
print(f"  Effect size (Cohen's h): {effect_size:.4f}")
print(f"  Tamaño de muestra por grupo: {int(np.ceil(n_per_group)):,}")
print(f"  Total de usuarios necesarios: {int(np.ceil(n_per_group * 2)):,}")

print("\n💡 Este es el mínimo necesario para detectar el efecto con confianza")

---

## 🧪 Ejercicio 2: Simular Experimento A/B

**Objetivo**: Generar datos sintéticos de un experimento A/B

**Escenarios**:
1. Grupo A (Control): 10% conversión
2. Grupo B (Tratamiento): 11% conversión

**Usuarios**: 5,000 por grupo

In [0]:
print("🧪 Ejercicio 2: Simular experimento A/B\n" + "="*60)

# Simular experimento
np.random.seed(42)
n_users_A = 5000
n_users_B = 5000

# Grupo A (Control): 10% conversion
conversions_A = np.random.binomial(1, 0.10, n_users_A)

# Grupo B (Tratamiento): 11% conversion (10% lift relativo)
conversions_B = np.random.binomial(1, 0.11, n_users_B)

print("✅ Experimento A/B simulado\n")
print(f"Grupo A (Control):")
print(f"  Usuarios: {n_users_A:,}")
print(f"  Conversiones: {conversions_A.sum():,}")
print(f"  Tasa de conversión: {conversions_A.mean():.4f} ({conversions_A.mean():.2%})")

print(f"\nGrupo B (Tratamiento):")
print(f"  Usuarios: {n_users_B:,}")
print(f"  Conversiones: {conversions_B.sum():,}")
print(f"  Tasa de conversión: {conversions_B.mean():.4f} ({conversions_B.mean():.2%})")

lift_observed = (conversions_B.mean() - conversions_A.mean()) / conversions_A.mean()
print(f"\nLift observado: {lift_observed:.2%}")

---

## 📈 Ejercicio 3: Test de Significancia Estadística

**Objetivo**: Determinar si la diferencia observada es estadísticamente significativa

**Método**:
* Test de proporciones Z (two-tailed)
* Hipótesis nula (H0): p_A = p_B
* Hipótesis alternativa (H1): p_A ≠ p_B
* Nivel de significancia: 0.05

**Decisión**: Si p-value < 0.05, rechazamos H0

In [0]:
print("📈 Ejercicio 3: Test de significancia\n" + "="*60)

# Test de proporciones
counts = [conversions_B.sum(), conversions_A.sum()]
nobs = [n_users_B, n_users_A]

z_stat, p_value = proportions_ztest(counts, nobs)

print("\nResultados del test:")
print(f"  Z-statistic: {z_stat:.4f}")
print(f"  P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Resultado: SIGNIFICATIVO")
    print("  • Rechazamos H0 (p_A = p_B)")
    print("  • B es estadísticamente mejor que A")
    lift = (conversions_B.mean() - conversions_A.mean()) / conversions_A.mean()
    print(f"  • Lift: {lift:.2%}")
    print(f"  • Diferencia absoluta: {(conversions_B.mean() - conversions_A.mean()):.4f}")
else:
    print("\n❌ Resultado: NO SIGNIFICATIVO")
    print("  • No podemos rechazar H0")
    print("  • No hay evidencia suficiente de diferencia")
    print("  • Necesitarías más datos o un efecto mayor")

print("\n💡 Interpretación:")
if p_value < 0.001:
    print("  P-value < 0.001: Evidencia muy fuerte")
elif p_value < 0.01:
    print("  P-value < 0.01: Evidencia fuerte")
elif p_value < 0.05:
    print("  P-value < 0.05: Evidencia moderada")
else:
    print("  P-value >= 0.05: Sin evidencia significativa")

---

## 💪 Ejercicio 4: Análisis de Poder Estadístico

**Objetivo**: Calcular el poder estadístico alcanzado con el tamaño de muestra usado

**Poder estadístico**:
* Probabilidad de detectar un efecto cuando realmente existe
* 1 - β (donde β es el error tipo II)
* Ideal: ≥ 0.80

**Escenarios**:
1. Poder con n=5,000 (lo que usamos)
2. Comparar con tamaño de muestra teórico

In [0]:
print("💪 Ejercicio 4: Análisis de poder estadístico\n" + "="*60)

# Calcular poder alcanzado con n=5,000
actual_power = zt_ind_solve_power(
    effect_size=effect_size,
    nobs1=n_users_A,
    alpha=0.05,
    ratio=1.0,
    alternative='two-sided'
)

print("\n📊 Poder estadístico alcanzado:")
print(f"  Con n={n_users_A:,} por grupo: {actual_power:.4f} ({actual_power:.1%})")

if actual_power >= 0.80:
    print(f"  ✅ Poder adecuado (≥ 80%)")
else:
    print(f"  ⚠️ Poder insuficiente (< 80%)")
    print(f"  Necesitarías {int(np.ceil(n_per_group)):,} usuarios por grupo")

print("\n📊 Comparación:")
print(f"  Tamaño de muestra teórico: {int(np.ceil(n_per_group)):,} por grupo")
print(f"  Tamaño usado en el experimento: {n_users_A:,} por grupo")

if n_users_A >= n_per_group:
    print("  ✅ Tamaño suficiente")
else:
    deficit = int(np.ceil(n_per_group)) - n_users_A
    print(f"  ⚠️ Faltan {deficit:,} usuarios por grupo para llegar al 80% de poder")

print("\n💡 Conclusión:")
if p_value < 0.05 and actual_power >= 0.80:
    print("  ✅ Experimento exitoso: Resultado significativo con poder adecuado")
    print("  🚀 Recomendación: Implementar variante B")
elif p_value < 0.05 and actual_power < 0.80:
    print("  ⚠️ Resultado significativo pero poder bajo")
    print("  🔄 Recomendación: Confirmar con más datos")
else:
    print("  ❌ Resultado no significativo")
    print("  🔄 Recomendación: Continuar experimento o aumentar efecto")

---

## ✅ Resumen de la Práctica

### 🎯 Conceptos Aprendidos

1. **Cálculo de Tamaño de Muestra**
   * Fórmula basada en effect size (Cohen's h)
   * Parámetros: α (significancia), 1-β (poder), MDE (mínimo efecto detectable)
   * Planificación antes de experimentar

2. **Test de Significancia**
   * Test de proporciones Z
   * Hipótesis nula vs alternativa
   * P-value y decisión estadística

3. **Poder Estadístico**
   * Probabilidad de detectar efecto real
   * Trade-off entre n, efecto y poder
   * Validación post-experimento

4. **Interpretación de Resultados**
   * Significancia estadística ≠ significancia práctica
   * Lift relativo vs absoluto
   * Decisión de negocio basada en datos

### 🚀 Próximos Pasos

* Implementar A/B testing en producción
* Monitoreo continuo de métricas
* Consideraciones de múltiples tests (corrección de Bonferroni)
* Experimentación secuencial y bandits

---

**Universidad del Aconcagua 🇦🇷**